In [1]:
#Abigail Birnbaum
#Last updated: 2026-10-01
#script to process HUC-2/PFAF regions and get shapefile for each

# Import Statements

In [1]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import xarray as xr

# Load in shapefiles

In [ ]:
pwp = '/gcfs/projects/foresight/hydrology/'
path_to_shps = pwp+'abirn_vic/input_data/initial_input_data/conus_HUC12_merit_v7d.gpkg'

In [3]:
basins_shp = gpd.read_file(path_to_shps)
basins_shp = basins_shp.filter(['HUCIDX','HUC_SRC','ToHUC','geometry']) #96715 unique
basins_shp.loc[(basins_shp.HUC_SRC=='WBDHU12'),'HUC2'] = basins_shp['HUCIDX'].str[0:2]
basins_shp.loc[(basins_shp.HUC_SRC=='MERIT_hydro'),'PFAF'] = basins_shp['HUCIDX'].str[0:2]

ERROR 1: PROJ: proj_create_from_database: Open of /home/birn696/.conda/envs/foresight_vic/share/proj failed


# Generate shapefile for each HUC-02 region

In [ ]:
#get list of unique HUC-2 regions and PFAF regions
huc2_list = np.sort(basins_shp[(basins_shp.HUC_SRC=='WBDHU12')].HUC2.unique())
pfaf_list = np.sort(basins_shp[(basins_shp.HUC_SRC=='MERIT_hydro')].PFAF.unique())

#path to save output
save_huc_path = pwp+'abirn_vic/input_data/ncar_region_shapefiles/'

for h2 in huc2_list:
    #filter shapefile to selected HUC-02 region
    basin_shp = basins_shp[basins_shp.HUC2==h2].reset_index().drop(columns=['index']) #filter to just HUC2 we're looking at
    #basin_shp.to_file(save_huc_path +'region_huc'+str(h2)+'.shp')
    
for pf in pfaf_list:
    #filter shapefile to selected pfaf region
    basin_shp = basins_shp[basins_shp.PFAF==pf].reset_index().drop(columns=['index']) #filter to just PFAF we're looking at
    #basin_shp.to_file(save_huc_path +'region_pfaf'+str(pf)+'.shp')

# Load in North America domain file from VIC-Global

In [ ]:
domain_path = pwp+'abirn_vic/input_data/initial_input_data/namerica_domain.nc'
na_domain = xr.open_dataset(domain_path)

### Create file to store all bounds (not including buffer)

In [ ]:
reg_list = ['huc01', 'huc02', 'huc03', 'huc04', 'huc05', 'huc06', 'huc07', 'huc08', 'huc09',
            'huc10', 'huc11', 'huc12', 'huc13', 'huc14', 'huc15','huc16', 'huc17', 'huc18',
            'pfaf71','pfaf72','pfaf73','pfaf75','pfaf77','pfaf78']

stor_bounds = np.empty([len(reg_list),12])

shp_path = pwp+'abirn_vic/input_data/ncar_region_shapefiles/'

#projected coordinate system to get bounds for
crs_proj_weather = '+proj=lcc +lat_0=40.0000076293945 +lon_0=-97 +lat_1=30 +lat_2=45 +x_0=0 +y_0=0 +R=6370000 +units=m +no_defs'
crs_proj_runoff = 'epsg:5070' #NAD83 CONUS Albers equal area- US CONUS

for i in range(len(reg_list)):
    reg_sel = reg_list[i]
    print(reg_sel)
    reg = gpd.read_file(shp_path+'region_'+reg_sel+'.shp')
    reg_proj_weather = reg.to_crs(crs_proj_weather)
    reg_proj_runoff = reg.to_crs(crs_proj_runoff)
    #min_lon,min_lat,max_lon,max_lat = huc2.total_bounds
    stor_bounds[i,0:4] = reg.total_bounds
    stor_bounds[i,4:8] = reg_proj_weather.total_bounds
    stor_bounds[i,8:] = reg_proj_runoff.total_bounds

huc01
huc02
huc03
huc04
huc05
huc06
huc07
huc08
huc09
huc10
huc11
huc12
huc13
huc14
huc15
huc16
huc17
huc18
pfaf71
pfaf72
pfaf73
pfaf75
pfaf77
pfaf78


In [24]:
stor_bounds_df = pd.DataFrame(stor_bounds,columns=['min_lon','min_lat','max_lon','max_lat',
                                                  'min_lon_weather','min_lat_weather','max_lon_weather','max_lat_weather',
                                                  'min_lon_runoff','min_lat_runoff','max_lon_runoff','max_lat_runoff'])
stor_bounds_df['region'] = reg_list

In [25]:
display(stor_bounds_df)

,min_lon,min_lat,max_lon,max_lat,min_lon_weather,min_lat_weather,max_lon_weather,max_lat_weather,min_lon_runoff,min_lat_runoff,max_lon_runoff,max_lat_runoff,region
0,-73.737982,40.939104,-66.875136,48.099706,1.899155e+06,3.436860e+05,2.329118e+06,1.222200e+06,1.824354e+06,2.209717e+06,2.258671e+06,3.086952e+06,huc01
1,-80.540992,36.669419,-71.850742,44.153047,1.434935e+06,-1.728241e+05,2.068718e+06,6.866254e+05,1.350627e+06,1.687621e+06,1.991476e+06,2.557120e+06,huc02
2,-90.623496,24.512236,-75.460832,37.521035,6.025990e+05,-1.596166e+06,1.918154e+06,-1.218288e+05,5.088583e+05,2.708342e+05,1.833348e+06,1.740462e+06,huc03
3,-93.213865,40.394681,-71.649333,48.358120,2.891115e+05,1.144124e+05,1.970226e+06,9.522573e+05,2.132424e+05,1.990595e+06,1.897904e+06,2.835502e+06,huc04
4,-89.266372,35.313316,-77.839370,42.449834,6.751534e+05,-4.543023e+05,1.566781e+06,4.193422e+05,5.888535e+05,1.413276e+06,1.488242e+06,2.292867e+06,huc05
5,-88.679600,34.111560,-81.261118,37.240546,7.318450e+05,-5.958250e+05,1.379104e+06,-1.942120e+05,6.450156e+05,1.272268e+06,1.294067e+06,1.672574e+06,huc06
6,-97.363941,36.949176,-85.892306,47.771651,-2.826801e+04,-3.090027e+05,9.187293e+05,8.655248e+05,-1.061297e+05,1.565495e+06,8.374919e+05,2.754077e+06,huc07
7,-94.338912,28.899084,-88.289406,37.861338,2.414950e+05,-1.197552e+06,7.803564e+05,-2.152126e+05,1.509770e+05,6.695203e+05,6.897786e+05,1.662002e+06,huc08
8,-114.087457,45.476081,-89.859297,50.552425,-1.255049e+06,6.064575e+05,5.281255e+05,1.195517e+06,-1.332135e+06,2.498744e+06,4.557103e+05,3.083409e+06,huc09
9,-113.938141,37.024602,-90.114221,49.739086,-1.310465e+06,-3.225779e+05,5.907819e+05,1.144034e+06,-1.389617e+06,1.556214e+06,5.058015e+05,3.039442e+06,huc10


In [ ]:
save_reg_bounds = pwp+'abirn_vic/input_data/extra_spatial_data/region_bounds.csv'
stor_bounds_df.to_csv(save_reg_bounds,index=False)